In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/niat-masterclass-rag-challenge/zyro-dynamics-hr-corpus/10_Travel_and_Expense_Policy.pdf
/kaggle/input/competitions/niat-masterclass-rag-challenge/zyro-dynamics-hr-corpus/03_Work_From_Home_Policy.pdf
/kaggle/input/competitions/niat-masterclass-rag-challenge/zyro-dynamics-hr-corpus/09_Onboarding_and_Separation_Policy.pdf
/kaggle/input/competitions/niat-masterclass-rag-challenge/zyro-dynamics-hr-corpus/07_IT_and_Data_Security_Policy.pdf
/kaggle/input/competitions/niat-masterclass-rag-challenge/zyro-dynamics-hr-corpus/06_Compensation_and_Benefits_Policy.pdf
/kaggle/input/competitions/niat-masterclass-rag-challenge/zyro-dynamics-hr-corpus/sample_submission.xlsx
/kaggle/input/competitions/niat-masterclass-rag-challenge/zyro-dynamics-hr-corpus/04_Code_of_Conduct.pdf
/kaggle/input/competitions/niat-masterclass-rag-challenge/zyro-dynamics-hr-corpus/01_Employee_Handbook.pdf
/kaggle/input/competitions/niat-masterclass-rag-challenge/zyro-dynamics-hr-corpus/05_Performance

# Zyro Dynamics HR Help Desk - RAG Challenge

This notebook builds a Retrieval-Augmented Generation (RAG) chatbot for answering employee HR policy questions using Zyro Dynamics policy documents.

### Steps
1. Install Dependencies
2. Load Documents
3. Create Chunks
4. Generate Embeddings
5. Build FAISS Vector Store
6. Create Retriever
7. Initialize LLM
8. Build RAG Chain
9. Add Guardrails
10. Generate Submission

In [2]:
!pip install -q \
langchain \
langchain-community \
langchain-core \
langchain-groq \
langchain-text-splitters \
faiss-cpu \
pypdf \
sentence-transformers \
cryptography \
openpyxl \
langsmith

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 9.2 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 554.9/554.9 kB 22.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 48.3 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 6.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 32.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 3.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.39.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
google-adk 1.29.0 requires google-cloud-bigquery-storage>=2.0.0, which is not installed.
ydata-profiling 4.18.4 requires numba<0.63,>=0.60, but you have numba 0.65.1 which is incompatible.
ydata-profiling 4.18.4 requires numpy<2.4,>=1.22,

# Imports and Dataset Configuration

Load all required libraries and define dataset paths.

# Step 1: Imports and Configuration

In this step, we import required libraries and define the corpus path where all HR policy PDFs are stored.

In [3]:
import os
import csv
import time
import pandas as pd

from cryptography.fernet import Fernet

from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

DATA_PATH = "/kaggle/input/competitions/niat-masterclass-rag-challenge/zyro-dynamics-hr-corpus"

print("Imports successful ✅")
print("DATA_PATH exists:", os.path.exists(DATA_PATH))
print("Files:", len(os.listdir(DATA_PATH)))

/tmp/ipykernel_58/2386448473.py:8: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


Imports successful ✅
DATA_PATH exists: True
Files: 13


# Step 2: Load HR Policy Documents

Load all PDF policy documents from the Zyro Dynamics HR corpus.

In [4]:
pdf_files = sorted([
    os.path.join(DATA_PATH, f)
    for f in os.listdir(DATA_PATH)
    if f.endswith(".pdf")
])

documents = []

for pdf_file in pdf_files:
    loader = PyPDFLoader(pdf_file)
    pages = loader.load()

    for page in pages:
        page.metadata["source"] = os.path.basename(pdf_file)

    documents.extend(pages)

print("Total PDF files loaded:", len(pdf_files))
print("Total pages loaded:", len(documents))

for file in pdf_files:
    print("Loaded:", os.path.basename(file))

Total PDF files loaded: 11
Total pages loaded: 39
Loaded: 00_Company_Profile.pdf
Loaded: 01_Employee_Handbook.pdf
Loaded: 02_Leave_Policy.pdf
Loaded: 03_Work_From_Home_Policy.pdf
Loaded: 04_Code_of_Conduct.pdf
Loaded: 05_Performance_Review_Policy.pdf
Loaded: 06_Compensation_and_Benefits_Policy.pdf
Loaded: 07_IT_and_Data_Security_Policy.pdf
Loaded: 08_Prevention_of_Sexual_Harassment_Policy.pdf
Loaded: 09_Onboarding_and_Separation_Policy.pdf
Loaded: 10_Travel_and_Expense_Policy.pdf


# Step 3: Split Documents into Chunks

Split the loaded PDF pages into smaller overlapping chunks for better retrieval accuracy.

In [6]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=100
)

chunks = text_splitter.split_documents(documents)

print("Total chunks created:", len(chunks))
print("\nSample chunk:")
print(chunks[0].page_content[:500])

Total chunks created: 168

Sample chunk:
Zyro Dynamics Pvt. Ltd.
Company Profile
Doc Code: ZDL-CORP-001
Confidential — For Internal Use Only
Page 1
 Zyro Dynamics Pvt. Ltd.
 
 Navigate the Future
 Company Profile
 Document Code
 ZDL-CORP-001
 Version
 V.01
 Effective Date
 01 April 2025
 Document Owner
 Corporate Communications
COMPANY OVERVIEW
Zyro Dynamics Pvt. Ltd. is a product-based enterprise software company headquartered in the United States. 
Founded in 2014 by Lamine Yamal and Aitana Bonmati, the company was built on a


# Step 4: Create Embeddings

Convert document chunks into vector embeddings using a Sentence Transformer model.

In [7]:
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True}
)

print("Embedding model loaded ✅")

/tmp/ipykernel_58/2561505423.py:1: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding model loaded ✅


# Step 5: Build FAISS Vector Store and Retriever

Store all chunk embeddings in a FAISS vector database and create an MMR retriever for accurate and diverse retrieval.

In [8]:
vectorstore = FAISS.from_documents(chunks, embeddings)

retriever = vectorstore.as_retriever(
    search_type="mmr",
    search_kwargs={
        "k": 5,
        "fetch_k": 20,
        "lambda_mult": 0.6
    }
)

print("FAISS vector store created ✅")
print("Total chunks indexed:", len(chunks))

FAISS vector store created ✅
Total chunks indexed: 168


# Step 6: Load API Keys and Configure LangSmith

Load Groq and LangSmith API keys from Kaggle Secrets and enable tracing.

In [11]:
from kaggle_secrets import UserSecretsClient

secrets = UserSecretsClient()

os.environ["GROQ_API_KEY"] = secrets.get_secret("GROQ_API_KEY")
os.environ["LANGCHAIN_API_KEY"] = secrets.get_secret("LANGCHAIN_API_KEY")
os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_PROJECT"] = "zyro-rag-challenge"

print("Groq and LangSmith configured ✅")

Groq and LangSmith configured ✅


# Step 7: Initialize Groq LLM

Load the Groq model that will answer questions using retrieved HR policy context.

In [12]:
llm = ChatGroq(
    model="llama-3.1-8b-instant",
    temperature=0
)

print("LLM initialized ✅")

LLM initialized ✅


# Step 8: Build RAG Chain and ask_bot Function

This step creates the final RAG pipeline and wraps it inside an ask_bot() function required for submission generation.

In [15]:
def format_docs(docs):
    return "\n\n".join(
        f"Source: {doc.metadata.get('source', 'Unknown')}\n{doc.page_content}"
        for doc in docs
    )

RAG_PROMPT = ChatPromptTemplate.from_template("""
You are Zyro Dynamics HR Help Desk Assistant.

Answer the employee's question using ONLY the HR policy context provided below.

Rules:
1. If the answer is not present in the context, say: "I can only answer questions based on Zyro Dynamics HR policy documents."
2. Do not use outside knowledge.
3. Keep the answer concise and policy-based.
4. Mention the relevant policy/source name when available.
5. For non-HR or out-of-scope questions, politely refuse.

HR Policy Context:
{context}

Employee Question:
{question}

Final Answer:
""")

rag_chain = (
    {
        "context": retriever | format_docs,
        "question": RunnablePassthrough()
    }
    | RAG_PROMPT
    | llm
    | StrOutputParser()
)

def ask_bot(question: str):
    answer = rag_chain.invoke(question)
    return {"answer": answer}

print("RAG chain and ask_bot() created ✅")

RAG chain and ask_bot() created ✅


# Step 9: Test the HR Chatbot

Test the chatbot with HR-related and out-of-scope questions.

In [16]:
test_questions = [
    "How many earned leaves are employees entitled to?",
    "What is the work from home policy?",
    "What is the capital of India?"
]

for q in test_questions:
    print("=" * 80)
    print("Question:", q)
    print("Answer:")
    print(ask_bot(q)["answer"])
    print()

Question: How many earned leaves are employees entitled to?
Answer:
According to the 02_Leave_Policy.pdf, employees become eligible for 15 days of Earned Leave upon completion of one year of continuous service.

Question: What is the work from home policy?
Answer:
According to the Work From Home Policy (ZDL-HR-003, V.02), Zyro Dynamics recognises that flexible work arrangements are an important part of employee wellbeing and has implemented a work from home policy to facilitate this.

Question: What is the capital of India?
Answer:
I can only answer questions based on Zyro Dynamics HR policy documents.



# Step 10: Load Evaluation Questions

Decrypt the hidden evaluation questions provided by the challenge.

In [18]:
import json
from cryptography.fernet import Fernet

SUBMISSION_SECRET = b"6Q_EBPtBG-60URcrF6jxNTJSRjy-CtZbJlvp_xf0c_M="
fernet = Fernet(SUBMISSION_SECRET)

starter_path = DATA_PATH + "/Starter_Notebook.ipynb"

with open(starter_path, "r", encoding="utf-8") as f:
    starter_nb = json.load(f)

cell_15_code = None

for cell in starter_nb["cells"]:
    source = "".join(cell.get("source", []))
    if "_Q = [" in source and "eval_questions" in source:
        cell_15_code = source
        break

exec(cell_15_code)

print("Evaluation questions are ready ✅")

15 evaluation questions loaded.
Evaluation questions are ready ✅


# Step 11: Generate Final Submission CSV

Generate encrypted answers for all evaluation questions and create the final submission.csv file.

In [19]:
import csv
import time

streamlit_link = "https://zyro-dynamics-hr-chatbot.streamlit.app"
langsmith_link = "https://smith.langchain.com/public/0f7ef6a0-b41d-4fc9-9ddf-59881acb5a2f/r"

rows = []

for i, q in enumerate(eval_questions):
    qid = q["question_id"]
    question = q["question"]

    try:
        result = ask_bot(question)
        answer = result["answer"]
        status = "OK"
    except Exception as e:
        answer = f"Error: {str(e)}"
        status = "ERROR"

    rows.append({
        "question_id": qid,
        "question_enc": fernet.encrypt(question.encode()).decode(),
        "answer_enc": fernet.encrypt(answer.encode()).decode(),
        "streamlit_link": streamlit_link,
        "langsmith_link": langsmith_link,
    })

    print(f"[{i+1:02d}/{len(eval_questions)}] {qid} ... {status}")

    if i < len(eval_questions) - 1:
        time.sleep(2)

with open("submission.csv", "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(
        f,
        fieldnames=[
            "question_id",
            "question_enc",
            "answer_enc",
            "streamlit_link",
            "langsmith_link"
        ]
    )
    writer.writeheader()
    writer.writerows(rows)

print("\nCorrect submission.csv generated ✅")

[01/15] Q01 ... OK
[02/15] Q02 ... OK
[03/15] Q03 ... OK
[04/15] Q04 ... OK
[05/15] Q05 ... OK
[06/15] Q06 ... OK
[07/15] Q07 ... OK
[08/15] Q08 ... OK
[09/15] Q09 ... OK
[10/15] Q10 ... OK
[11/15] Q11 ... OK
[12/15] Q12 ... OK
[13/15] Q13 ... OK
[14/15] Q14 ... OK
[15/15] Q15 ... OK

Correct submission.csv generated ✅


# Step 12: Verify Generated Submission

Verify that the generated submission.csv contains all required rows and columns before final submission.

In [20]:
import pandas as pd

submission = pd.read_csv("submission.csv")

print("Rows:", len(submission))
print("Columns:", submission.columns.tolist())

display(submission.head())

Rows: 15
Columns: ['question_id', 'question_enc', 'answer_enc', 'streamlit_link', 'langsmith_link']


,question_id,question_enc,answer_enc,streamlit_link,langsmith_link
0,Q01,gAAAAABqLvTVjf0BdniBuZzCSTbevdrzVZHErBtcaB196Q...,gAAAAABqLvTVvWXt8m7W7osganPd9KhWzsVL2fWxwlXpOm...,https://zyro-dynamics-hr-chatbot.streamlit.app,https://smith.langchain.com/public/0f7ef6a0-b4...
1,Q02,gAAAAABqLvTXP0goq1skNVlWaZ3CVSMtBQrVONzGqc1IC7...,gAAAAABqLvTXC6eH4z7Z23203LKusCJKv8Jn0wjaRocqD8...,https://zyro-dynamics-hr-chatbot.streamlit.app,https://smith.langchain.com/public/0f7ef6a0-b4...
2,Q03,gAAAAABqLvTap1eQfTC8kiGMq8kqQHu59DzgNYflq_z-h6...,gAAAAABqLvTar8V-bQicVmwC7Uypf5mYxqcLQvD9TJhIjL...,https://zyro-dynamics-hr-chatbot.streamlit.app,https://smith.langchain.com/public/0f7ef6a0-b4...
3,Q04,gAAAAABqLvTcFE3Pj1yigWlidRisq0MjI2mYvjT9_ELAuy...,gAAAAABqLvTc1PAv0-N-yy5_X9wWafpaQyIyyUAzyAosYd...,https://zyro-dynamics-hr-chatbot.streamlit.app,https://smith.langchain.com/public/0f7ef6a0-b4...
4,Q05,gAAAAABqLvTeY5BJJX0esbliv3pxinjmzgYNkSg0cWxrBw...,gAAAAABqLvTewUr1eLSBBG-OPB-MbrAaCVYQNc2iYWWxyq...,https://zyro-dynamics-hr-chatbot.streamlit.app,https://smith.langchain.com/public/0f7ef6a0-b4...


# Step 14: Final Validation

Perform one final validation before saving the notebook version and submitting to Kaggle.

In [23]:
import os
import pandas as pd

assert os.path.exists("submission.csv")

df = pd.read_csv("submission.csv")

assert len(df) == 15

required_cols = [
    "question_id",
    "question_enc",
    "answer_enc",
    "streamlit_link",
    "langsmith_link"
]

for col in required_cols:
    assert col in df.columns

print("Submission validation passed ✅")
print("Rows:", len(df))
print("Columns:", len(df.columns))

Submission validation passed ✅
Rows: 15
Columns: 5
